# RxNorm collection from DSLD — preview before saving
Independent RxNorm collector for active ingredient rows, nested rows and forms in the ten supplement categories' DSLD labels. Includes co-ingredients; excludes `otheringredients` excipients. Category membership records the original DSLD search, not a verified ingredient classification.

Search active RxNorm concepts (`allsrc=0`) by exact name first (`search=0`), then normalized name (`search=1`) only on no match. Normalization can ignore salts, word order and punctuation: those candidates explicitly require review. No approximate/fuzzy searches or automatic PubChem cross-mapping. Do not interpret absent matches as absence of an ingredient or evidence of safety.

Collect public concept properties, names, attributes and codes. This is terminology data, not interaction or clinical evidence. New responses stay in memory until the final save cell is enabled after preview.

References:
- https://www.lhncbc.nlm.nih.gov/RxNav/APIs/api-RxNorm.findRxcuiByString.html
- https://lhncbc.nlm.nih.gov/RxNav/APIs/api-RxNorm.getRxConceptProperties.html
- https://lhncbc.nlm.nih.gov/RxNav/APIs/api-RxNorm.getAllProperties.html
- https://www.lhncbc.nlm.nih.gov/RxNav/TermsofService.html


In [1]:
import json
import time
import hashlib
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from email.utils import parsedate_to_datetime
import requests

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DSLD_DIR = PROJECT_ROOT / 'data/raw/dsld'
RAW_DIR = PROJECT_ROOT / 'data/raw/rxnorm'
BASE_URL = 'https://rxnav.nlm.nih.gov/REST'
SUPPLEMENTS = ['Magnesium', 'Vitamin D', 'Ashwagandha', 'Melatonin', 'Omega-3',
               'Vitamin B12', 'Iron', 'Turmeric', 'Probiotics', 'Creatine']
MIN_INTERVAL = 0.15  # Below RxNav's published 20 requests/second/IP limit.
MAX_ATTEMPTS = 3
SAVE_DATA = False
session = requests.Session()
session.headers['User-Agent'] = 'SupplementRx-AI-Academic-POC/1.0'
def utcnow():
    return datetime.now(timezone.utc).isoformat()
# Reset in-memory state when rerunning configuration. Resume by rerunning later cells only.
search_results = {}
records = {}
collection_started_at = utcnow()


## 1. Discover terms directly from DSLD
Preserve original names, ingredient IDs, row paths, category provenance and input-file checksums. Case and whitespace grouping deduplicates HTTP queries without asserting biomedical equivalence. This notebook does not depend on running the PubChem notebook.


In [2]:
def walk_rows(rows, path='ingredientRows'):
    for i, row in enumerate(rows or []):
        row_path = f'{path}[{i}]'
        yield row, row_path
        yield from walk_rows(row.get('nestedRows'), row_path + '.nestedRows')

def build_inventory():
    manifest_bytes = (DSLD_DIR / 'collection_manifest.json').read_bytes()
    manifest = json.loads(manifest_bytes)
    groups = {}
    label_hashes = {}
    for label_id, categories in manifest['category_matches'].items():
        categories = sorted(set(categories) & set(SUPPLEMENTS))
        if not categories:
            continue
        path = DSLD_DIR / 'labels' / f'label_{label_id}.json'
        raw = path.read_bytes()  # Missing/invalid source records fail explicitly.
        label_hashes[label_id] = hashlib.sha256(raw).hexdigest()
        label = json.loads(raw)
        if str(label['id']) != label_id:
            raise ValueError(f'Label ID mismatch: {path}')
        for row, row_path in walk_rows(label.get('ingredientRows')):
            objects = [(row, row_path, 'ingredient')]
            objects += [(form, f'{row_path}.forms[{i}]', 'form')
                        for i, form in enumerate(row.get('forms') or [])]
            for obj, obj_path, kind in objects:
                original = obj.get('name')
                if not original or not original.strip():
                    continue
                key = ' '.join(original.casefold().split())
                group = groups.setdefault(key, {'query': original.strip(), 'variants': [],
                                               'occurrences': []})
                if original not in group['variants']:
                    group['variants'].append(original)
                group['occurrences'].append({
                    'dsld_id': label_id, 'search_categories': categories,
                    'ingredient_id': obj.get('ingredientId'), 'name': original,
                    'kind': kind, 'json_path': obj_path})
    provenance = {'manifest_sha256': hashlib.sha256(manifest_bytes).hexdigest(),
                  'label_sha256': label_hashes}
    return dict(sorted(groups.items())), provenance

query_inventory, dsld_provenance = build_inventory()
print('Source labels:', len(dsld_provenance['label_sha256']))
print('Distinct original names:', len({v for g in query_inventory.values() for v in g['variants']}))
print('Query groups:', len(query_inventory))


Source labels: 400
Distinct original names: 753
Query groups: 739


## 2. HTTP client and response validation
Record raw JSON, URLs, parameters, status codes, request times and retry attempts. Only a successful search with an empty `idGroup.rxnormId` is a no-match result. HTTP errors and malformed payloads remain errors. Respect Retry-After and retry transient failures.


In [3]:
_last_request = 0.0

def request_json(path, params=None):
    global _last_request
    history = []
    url = f'{BASE_URL}/{path}'
    for attempt in range(1, MAX_ATTEMPTS + 1):
        time.sleep(max(0, MIN_INTERVAL - (time.monotonic() - _last_request)))
        _last_request = time.monotonic()
        item = {'url': url, 'params': params or {}, 'requested_at_utc': utcnow(), 'attempt': attempt}
        delay = 2 ** attempt
        retryable = True
        try:
            response = session.get(url, params=params, timeout=(10, 35))
            item.update(url=response.url, status_code=response.status_code)
            retry_after = response.headers.get('Retry-After')
            if retry_after:
                try:
                    delay = max(delay, float(retry_after))
                except ValueError:
                    try:
                        delay = max(delay, (parsedate_to_datetime(retry_after) - datetime.now(timezone.utc)).total_seconds())
                    except (ValueError, TypeError):
                        pass
            try:
                payload = response.json()
                item['response'] = payload
            except ValueError:
                item['response_text'] = response.text
                raise ValueError('Non-JSON response')
            if response.status_code == 200:
                if not isinstance(payload, dict) or 'error' in payload:
                    raise ValueError('Unexpected API payload')
                history.append(item)
                return {'status': 'ok', 'response': payload, 'requests': history}
            item['error'] = f'HTTP {response.status_code}'
            retryable = response.status_code in (429, 500, 502, 503, 504)
        except (requests.RequestException, ValueError) as exc:
            item['error'] = str(exc)
        history.append(item)
        if not retryable:
            break
        if attempt < MAX_ATTEMPTS:
            time.sleep(delay)
    return {'status': 'error', 'error': history[-1]['error'], 'requests': history}

def parse_search(response):
    if response['status'] != 'ok':
        return 'error', []
    payload = response['response']
    if 'idGroup' not in payload or not isinstance(payload['idGroup'], (dict, type(None))):
        response['validation_error'] = 'Missing or invalid idGroup'
        return 'error', []
    ids = (payload['idGroup'] or {}).get('rxnormId')
    if ids is None or ids == []:
        return 'not_found', []
    if not isinstance(ids, list) or not all(isinstance(x, str) and x.isdigit() for x in ids):
        response['validation_error'] = 'Invalid RxCUI list'
        return 'error', []
    return 'matched', sorted(set(ids), key=int)

def search_rxnorm(term):
    searches = []
    for mode, precision in [('exact', 0), ('normalized', 1)]:
        response = request_json('rxcui.json', {'name': term, 'allsrc': 0, 'search': precision})
        status, ids = parse_search(response)
        searches.append({'mode': mode, **response})
        if status != 'not_found':
            return {'status': status, 'match_method': mode if ids else None,
                    'rxcuis': ids, 'searches': searches,
                    'requires_identity_review': True}
    return {'status': 'not_found', 'match_method': None, 'rxcuis': [], 'searches': searches,
            'requires_identity_review': False}


## 3. Record source version and search inventory
Completed in-memory searches are reused when this cell alone is rerun; errors are retried. Preserve every candidate, including ambiguous matches. Errors in exact search do not silently trigger a fallback.


In [4]:
version_start = request_json('version.json')
if version_start['status'] != 'ok' or not version_start['response'].get('version'):
    raise RuntimeError('Cannot establish RxNorm release version; inspect version_start')
print('RxNorm version:', version_start['response'])
for i, (key, group) in enumerate(query_inventory.items(), 1):
    if search_results.get(key, {}).get('status') not in ('matched', 'not_found'):
        search_results[key] = search_rxnorm(group['query'])
    if i % 50 == 0 or i == len(query_inventory):
        print(f'Searched {i}/{len(query_inventory)}: '
              f'{dict(Counter(r["status"] for r in search_results.values()))}', flush=True)
candidate_rxcuis = sorted({rid for k in query_inventory for rid in search_results[k]['rxcuis']}, key=int)
print('Unique candidate RxCUIs:', len(candidate_rxcuis))


RxNorm version: {'version': '05-Oct-2026', 'apiVersion': '3.1.356'}
Searched 50/739: {'matched': 22, 'not_found': 28}
Searched 100/739: {'matched': 42, 'not_found': 58}
Searched 150/739: {'matched': 64, 'not_found': 86}
Searched 200/739: {'matched': 88, 'not_found': 112}
Searched 250/739: {'matched': 109, 'not_found': 141}
Searched 300/739: {'matched': 133, 'not_found': 167}
Searched 350/739: {'matched': 159, 'not_found': 191}
Searched 400/739: {'matched': 180, 'not_found': 220}
Searched 450/739: {'matched': 205, 'not_found': 245}
Searched 500/739: {'matched': 224, 'not_found': 276}
Searched 550/739: {'matched': 241, 'not_found': 309}
Searched 600/739: {'matched': 258, 'not_found': 342}
Searched 650/739: {'matched': 277, 'not_found': 373}
Searched 700/739: {'matched': 290, 'not_found': 410}
Searched 739/739: {'matched': 313, 'not_found': 426}
Unique candidate RxCUIs: 287


## 4. Download each unique concept once
Fetch canonical concept properties (including term type) and all public property categories (names, attributes and codes). Do not treat source-code entries as independently verified cross-database mappings. Empty or malformed concept responses are explicitly marked incomplete. Rerunning retries incomplete endpoints.


In [5]:
def validate_detail(kind, response, rxcui):
    if response['status'] != 'ok':
        return False
    payload = response['response']
    if kind == 'properties':
        prop = payload.get('properties') or {}
        valid = (isinstance(prop, dict) and prop.get('rxcui') == rxcui
                 and bool(prop.get('name')) and bool(prop.get('tty')))
    else:
        group = payload.get('propConceptGroup') or {}
        props = group.get('propConcept') if isinstance(group, dict) else None
        valid = (isinstance(props, list) and bool(props)
                 and all(isinstance(p, dict) and all(k in p for k in ('propName', 'propValue', 'propCategory'))
                         for p in props))
    if not valid:
        response['validation_error'] = f'Invalid or absent {kind} for RxCUI {rxcui}'
    return bool(valid)

for i, rid in enumerate(candidate_rxcuis, 1):
    record = records.setdefault(rid, {'rxcui': rid})
    for kind, endpoint, params in [('properties', 'properties.json', None),
                                   ('all_properties', 'allProperties.json', {'prop': 'ALL'})]:
        if not record.get(kind, {}).get('valid'):
            response = request_json(f'rxcui/{rid}/{endpoint}', params)
            response['valid'] = validate_detail(kind, response, rid)
            record[kind] = response
    if i % 25 == 0 or i == len(candidate_rxcuis):
        print(f'Fetched {i}/{len(candidate_rxcuis)} concepts', flush=True)
version_end = request_json('version.json')


Fetched 25/287 concepts
Fetched 50/287 concepts
Fetched 75/287 concepts
Fetched 100/287 concepts
Fetched 125/287 concepts
Fetched 150/287 concepts
Fetched 175/287 concepts
Fetched 200/287 concepts
Fetched 225/287 concepts
Fetched 250/287 concepts
Fetched 275/287 concepts
Fetched 287/287 concepts


## 5. Inspect coverage, completeness and size BEFORE saving
Exact and normalized matches remain separate. Counts by search category overlap because shared ingredient terms are queried only once. Full compound/medication identity verification is deferred to preprocessing. “Complete” below describes collection of selected API fields, not complete supplement coverage.


In [6]:
def make_preview():
    categories = {}
    for category in SUPPLEMENTS:
        keys = [k for k,g in query_inventory.items() if any(category in o['search_categories'] for o in g['occurrences'])]
        categories[category] = {
            'queries': len(keys),
            'exact_matches': sum(search_results[k]['match_method'] == 'exact' for k in keys),
            'normalized_matches': sum(search_results[k]['match_method'] == 'normalized' for k in keys),
            'no_match': sum(search_results[k]['status'] == 'not_found' for k in keys),
            'errors': sum(search_results[k]['status'] == 'error' for k in keys),
            'unique_candidate_rxcuis': len({rid for k in keys for rid in search_results[k]['rxcuis']})}
    complete = [rid for rid in candidate_rxcuis if all(records[rid].get(k, {}).get('valid')
                                                     for k in ('properties', 'all_properties'))]
    property_rows = [p for rid in candidate_rxcuis for p in
                     ((records[rid].get('all_properties', {}).get('response', {}).get('propConceptGroup') or {}).get('propConcept') or [])]
    version_stable = (version_end['status'] == 'ok' and version_start['response'] == version_end.get('response'))
    report = {
        'source_labels': len(dsld_provenance['label_sha256']), 'query_groups': len(query_inventory),
        'search_status': dict(Counter(search_results[k]['status'] for k in query_inventory)),
        'match_methods': dict(Counter(search_results[k]['match_method'] for k in query_inventory if search_results[k]['status']=='matched')),
        'ambiguous_queries': sum(len(search_results[k]['rxcuis']) > 1 for k in query_inventory),
        'unique_candidate_rxcuis': len(candidate_rxcuis), 'complete_concepts': len(complete),
        'incomplete_rxcuis': sorted(set(candidate_rxcuis) - set(complete), key=int),
        'concept_term_types': dict(Counter(records[rid].get('properties', {}).get('response', {}).get('properties', {}).get('tty', 'missing') for rid in candidate_rxcuis)),
        'property_entries_by_category': dict(Counter(p['propCategory'] for p in property_rows)),
        'source_version_stable': version_stable, 'by_category': categories}
    bundle = {'source': 'RxNorm public REST API', 'started_at_utc': collection_started_at,
              'finished_at_utc': utcnow(), 'version_start': version_start, 'version_end': version_end,
              'search_policy': {'allsrc':0, 'exact_then_normalized':True, 'approximate':False},
              'scope':'DSLD ingredient rows, nested ingredients and forms, including co-ingredients; excluding excipients',
              'dsld_provenance':dsld_provenance, 'queries':query_inventory,
              'search_results':{k:search_results[k] for k in query_inventory},
              'records':{rid:records[rid] for rid in candidate_rxcuis}}
    report['payload_bytes_before_report'] = len(json.dumps(bundle, ensure_ascii=False).encode('utf-8'))
    bundle['report'] = report
    return report, bundle

preview_report, collection_bundle = make_preview()
print(json.dumps(preview_report, indent=2))
print('Approximate payload MiB:', round(preview_report['payload_bytes_before_report']/1024**2, 3))
print('Normalized fallback candidates (review required):')
for key, result in search_results.items():
    if result['match_method'] == 'normalized':
        print(query_inventory[key]['query'], '->', [(rid, records[rid].get('properties',{}).get('response',{}).get('properties',{}).get('name')) for rid in result['rxcuis']])
print('No new RxNorm data has been saved.')


{
  "source_labels": 400,
  "query_groups": 739,
  "search_status": {
    "matched": 313,
    "not_found": 426
  },
  "match_methods": {
    "exact": 282,
    "normalized": 31
  },
  "ambiguous_queries": 31,
  "unique_candidate_rxcuis": 287,
  "complete_concepts": 287,
  "incomplete_rxcuis": [],
  "concept_term_types": {
    "IN": 256,
    "PIN": 30,
    "MIN": 1
  },
  "property_entries_by_category": {
    "ATTRIBUTES": 989,
    "CODES": 177517,
    "NAMES": 640,
    "SOURCES": 1601
  },
  "source_version_stable": true,
  "by_category": {
    "Magnesium": {
      "queries": 52,
      "exact_matches": 34,
      "normalized_matches": 0,
      "no_match": 18,
      "errors": 0,
      "unique_candidate_rxcuis": 35
    },
    "Vitamin D": {
      "queries": 58,
      "exact_matches": 38,
      "normalized_matches": 2,
      "no_match": 18,
      "errors": 0,
      "unique_candidate_rxcuis": 38
    },
    "Ashwagandha": {
      "queries": 40,
      "exact_matches": 13,
      "normalized_mat

## 6. Save only after reviewing the preview
Enable `SAVE_DATA` after checking the preceding output. Preserve all queries and raw request/response envelopes in an immutable run directory. The manifest records checksums, counts, source-version consistency and completeness; no existing source files are overwritten.


In [ ]:
# SAVE_DATA = True
if SAVE_DATA:
    run_dir = RAW_DIR / 'runs' / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    run_dir.mkdir(parents=True, exist_ok=False)
    payload = json.dumps(collection_bundle, ensure_ascii=False, indent=2).encode('utf-8')
    (run_dir/'collection.json').write_bytes(payload)
    manifest = {'source':'RxNorm', 'saved_at_utc':utcnow(), 'collection_file':'collection.json',
                'bytes':len(payload), 'sha256':hashlib.sha256(payload).hexdigest(),
                'complete': not preview_report['incomplete_rxcuis'] and not preview_report['search_status'].get('error',0) and preview_report['source_version_stable'],
                'report':preview_report}
    (run_dir/'collection_manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
    print('Saved:',run_dir)
else:
    print('Preview only. SAVE_DATA is False; responses remain in memory.')


## Executed collection result

Inspected coverage and size before saving to `data/raw/rxnorm/runs/20261028T060819025094Z`. The saved JSON is 72,476,471 bytes (69.12 MiB) with a verified SHA-256 checksum. Queried 739 groups from 400 DSLD labels: 282 exact matches, 31 normalized matches and 426 no matches, with zero request errors. Downloaded 287 unique candidate concepts (256 IN, 30 PIN, 1 MIN), each with valid properties and all-properties responses. There are 31 multi-candidate queries; those are a separate count from the 31 normalized queries. Source release remained 05-Oct-2026 throughout the run.

Normalized matches can differ materially from the source ingredient form and remain flagged for review. This is raw terminology collection, not identity resolution or interaction evidence. `SAVE_DATA` remains `False` for future runs.
